In [1]:
import os
from pydantic_ai import Agent
from pydantic_ai.capabilities import MCP
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

from IPython.display import display, Markdown

## Let us go one step higher in the abstraction by using Agents and letting them decide which tools to call rather than specifying them manually

In [2]:
agent = Agent(
    name="hpc-docs-agent",
    model=OpenAIResponsesModel(
        model_name="@vertexai/gemini-3.7-flash",
        provider=OpenAIProvider(
            base_url="https://ai-gateway.apps.cloud.rt.nyu.edu/v1/",
            api_key=os.getenv("PORTKEY_API_KEY"),
        ),
    ),
    instructions="Be concise and answer questions from retreived knowledge with tools.",
    capabilities=[
        MCP(
            url="https://mcp-gateway.apps.cloud.rt.nyu.edu/rts-docs-algolia-public/mcp",
            id="rts-docs-algolia-mcp",
            headers={
                "x-portkey-api-key": os.getenv("PORTKEY_API_KEY"),
            },
        ),
    ],
)

In [3]:
result = await agent.run("Login to HPC cluster from off campus") # await is added because the agent is run asynchronously

In [4]:
display(Markdown(result.output))

To log into the NYU HPC cluster (Torch) from off-campus, follow these steps:

---

### Step 1: Connect to the NYU VPN
Access to the cluster requires being on the NYU network. When off-campus, you must first connect via the **NYU VPN**:
* Set up and connect using the [NYU VPN client](https://www.nyu.edu/life/information-technology/infrastructure/network-services/vpn.html).

---

### Step 2: Connect to the Cluster

You can connect either via **Command Line (SSH)** or via the **Web Gateway (Open OnDemand)**:

#### Option A: Command Line (SSH)
1. Open your terminal (Terminal on macOS/Linux, PowerShell / WSL / PuTTY / MobaXterm on Windows).
2. Run the SSH command:
   ```bash
   ssh <NetID>@login.torch.hpc.nyu.edu
   ```
3. Complete the **Two-Factor Authentication (2FA)** prompt:
   * A PIN and link (`https://microsoft.com/devicelogin`) will appear in the terminal.
   * Open the link, enter the PIN, log in with your NYU credentials, and approve the Duo MFA push.
   * Return to your terminal and press `Enter`.

#### Option B: Web Browser (Open OnDemand)
* While connected to the VPN, navigate to **[https://ood.torch.hpc.nyu.edu](https://ood.torch.hpc.nyu.edu)**.
* Log in with your NYU credentials to access the web shell, Jupyter Notebooks, RStudio, and remote desktop environments directly from your browser.

## Let's see how we can evaluate the performance of the Agent on this task by varying the LLM used. 
## Here we check if the output from the Agent contains the specific string `login.torch.hpc.nyu.edu` to ensure that the model did not hallucinate a new login address:

In [5]:
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Contains

# Create a dataset with test cases
dataset = Dataset(
    name='login-to-hpc',
    cases=[
        Case(
            name="use-gemini-3.5-flash",
            inputs={
                "query": "Login to HPC cluster from off campus",
                "model": "@vertexai/gemini-3.7-flash"
            },
        ),
        Case(
            name="use-gemini-2.5-flash-lite",
            inputs={
                "query": "Login to HPC cluster from off campus",
                "model": "@vertexai/gemini-2.5-flash-lite"
            },
        ),
    ],
    evaluators=[
        Contains(value='login.torch.hpc.nyu.edu', case_sensitive=True),
    ],
)

async def agent_task(inputs: dict) -> str:
    with agent.override(model=
                        OpenAIResponsesModel(
                            model_name=inputs["model"],
                            provider=OpenAIProvider(
                                base_url="https://ai-gateway.apps.cloud.rt.nyu.edu/v1/",
                                api_key=os.getenv("PORTKEY_API_KEY"),
                                ),
                        )
                       ):
        result = await agent.run(user_prompt=inputs["query"])
        return result.output


# Run the evaluation
report = await dataset.evaluate(agent_task)

# Print the results
report.print()

Output()

           Evaluation Summary: agent_task            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Case ID                   ┃ Assertions ┃ Duration ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━┩
│ use-gemini-3.5-flash      │ ✔          │    13.1s │
├───────────────────────────┼────────────┼──────────┤
│ use-gemini-2.5-flash-lite │ ✗          │     4.2s │
├───────────────────────────┼────────────┼──────────┤
│ Averages                  │ 50.0% ✔    │     8.7s │
└───────────────────────────┴────────────┴──────────┘

## Why did the case with `gemini-2.5-flash-lite` fail? Let's check the output from that run:

In [6]:
display(Markdown(report.cases[1].output))

You'll need to be connected to the VPN to log in to the HPC cluster from off-campus. Once connected, you can log in using SSH.

Here are the general steps:

1.  **Connect to the VPN:** Follow the instructions for your operating system to connect to the NYU VPN.
2.  **Open a terminal or SSH client.**
3.  **Log in using SSH:**
    ```bash
    ssh your_netid@hpc.nyu.edu
    ```
    Replace `your_netid` with your NYU NetID.

If you encounter any issues, you can refer to the documentation for specific instructions and troubleshooting: [HPC bursting to cloud - Visualization Workstations](https://services.rt.nyu.edu/docs/cloud/hpc_bursting_to_cloud/visualization/)

This document provides detailed steps, including setting up your SSH config, requesting an interactive session, and using VNC for visualization purposes if needed.

## The older, less capable model misunderstood the prompt and answered the question for a different HPC cluster (Cloud bursting). Explore how you may prevent this?